# Gray-Scott Reaction-Diffusion

$$\frac{\partial U}{\partial t} = D_u \nabla^2 U - UV^2 + F(1 - U)$$

$$\frac{\partial V}{\partial t} = D_v \nabla^2 V + UV^2 - (F + k)V$$

Use the sliders below to explore different patterns by varying **F** (feed rate) and **k** (kill rate).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# tinygrad is a minimal tensor library (like PyTorch but ~10k lines of code).
# On Apple Silicon, it uses the Metal GPU backend automatically.
# Tensor operations (add, mul, conv2d) become GPU compute kernels.
from tinygrad import Tensor, dtypes
import time

In [2]:
def gray_scott(N=1024, F=0.04, k=0.06, Du=0.16, Dv=0.08, dt=1.0, steps=10000):
    """Run Gray-Scott on the GPU via tinygrad and return final U, V as numpy arrays."""

    # 9-point Laplacian kernel — weights diagonal neighbors at 0.05 so diffusion
    # spreads isotropically (not biased along grid axes like the simpler 5-point version).
    # The weights sum to zero: 4*0.2 + 4*0.05 - 1.0 = 0
    lap_np = np.array([[0.05, 0.2, 0.05],
                       [0.2, -1.0, 0.2],
                       [0.05, 0.2, 0.05]], dtype=np.float32)

    # conv2d expects shape (out_channels, in_channels, H, W)
    # We have 1 input channel and 1 output channel — just the 2D grid.
    lap_k = Tensor(lap_np.reshape(1, 1, 3, 3))

    # Initial conditions: U=1 (substrate full), V=0 (no autocatalyst)
    U_np = np.ones((N, N), dtype=np.float32)
    V_np = np.zeros((N, N), dtype=np.float32)

    # Seed a square patch in the center with V present and U reduced.
    # Small random noise breaks the symmetry so patterns aren't perfectly square.
    r = N // 10
    cx, cy = N // 2, N // 2
    U_np[cx-r:cx+r, cy-r:cy+r] = 0.50
    V_np[cx-r:cx+r, cy-r:cy+r] = 0.25
    U_np[cx-r:cx+r, cy-r:cy+r] += 0.02 * np.random.randn(2*r, 2*r).astype(np.float32)
    V_np[cx-r:cx+r, cy-r:cy+r] += 0.02 * np.random.randn(2*r, 2*r).astype(np.float32)

    # Move grids to GPU. Shape (1,1,N,N) is needed for conv2d.
    U = Tensor(U_np.reshape(1, 1, N, N))
    V = Tensor(V_np.reshape(1, 1, N, N))

    t0 = time.time()
    for i in range(steps):
        # Laplacian via convolution. padding=1 keeps the grid the same size.
        # This is where the GPU does the heavy lifting — each conv2d is a
        # single Metal compute kernel over the entire NxN grid in parallel.
        Lu = U.conv2d(lap_k, padding=1)
        Lv = V.conv2d(lap_k, padding=1)

        # The reaction: U*V*V is the autocatalytic term.
        # V consumes U to make more V, then V decays at rate (F+k).
        uvv = U * V * V
        U = (U + dt * (Du * Lu - uvv + F * (1.0 - U))).realize()
        V = (V + dt * (Dv * Lv + uvv - (F + k) * V)).realize()

        # .realize() forces tinygrad to actually run the computation NOW.
        # Without it, tinygrad lazily builds a computation graph and only
        # executes when you read a value. Calling realize() each step keeps
        # memory bounded — otherwise the graph would grow to millions of nodes.

    elapsed = time.time() - t0
    print(f"{steps} steps on {N}x{N} grid in {elapsed:.1f}s ({steps/elapsed:.0f} steps/s)")

    # Pull results back to CPU as numpy arrays for plotting
    return U.numpy().squeeze(), V.numpy().squeeze()

In [3]:
# Interactive explorer

f_slider = widgets.FloatSlider(
    value=0.04, min=0.01, max=0.08, step=0.002,
    description='F (feed):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
    readout_format='.3f',
)

k_slider = widgets.FloatSlider(
    value=0.06, min=0.03, max=0.07, step=0.002,
    description='k (kill):',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
    readout_format='.3f',
)

steps_slider = widgets.IntSlider(
    value=10000, min=1000, max=30000, step=1000,
    description='Steps:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

grid_slider = widgets.IntSlider(
    value=1024, min=256, max=4096, step=256,
    description='Grid size N:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

run_button = widgets.Button(description='Run simulation', button_style='primary')
output = widgets.Output()

presets_dropdown = widgets.Dropdown(
    options=[
        ('-- presets --', None),
        ('Mitosis (F=0.028, k=0.062)', (0.028, 0.062)),
        ('Coral growth (F=0.060, k=0.062)', (0.060, 0.062)),
        ('Spots (F=0.030, k=0.060)', (0.030, 0.060)),
        ('Stripes (F=0.040, k=0.060)', (0.040, 0.060)),
        ('Waves (F=0.014, k=0.045)', (0.014, 0.045)),
        ('Chaos (F=0.026, k=0.051)', (0.026, 0.051)),
    ],
    description='Preset:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px'),
)

def on_preset_change(change):
    if change['new'] is not None:
        f_slider.value, k_slider.value = change['new']

presets_dropdown.observe(on_preset_change, names='value')

def on_run(b):
    output.clear_output(wait=True)
    with output:
        F, k, N = f_slider.value, k_slider.value, grid_slider.value
        print(f'Running: F={F:.3f}, k={k:.3f}, N={N}, steps={steps_slider.value}...')
        U, V = gray_scott(N=N, F=F, k=k, steps=steps_slider.value)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
        im1 = ax1.imshow(U, cmap='viridis', vmin=0, vmax=1)
        ax1.set_title(f'U (substrate)\nF={F:.3f}, k={k:.3f}, N={N}')
        ax1.set_xlabel('x')
        ax1.set_ylabel('y')
        fig.colorbar(im1, ax=ax1, shrink=0.8)
        im2 = ax2.imshow(V, cmap='inferno', vmin=0, vmax=0.5)
        ax2.set_title(f'V (autocatalyst)\nF={F:.3f}, k={k:.3f}, N={N}')
        ax2.set_xlabel('x')
        ax2.set_ylabel('y')
        fig.colorbar(im2, ax=ax2, shrink=0.8)
        plt.tight_layout()
        plt.show()

run_button.on_click(on_run)

display(
    widgets.VBox([
        presets_dropdown,
        f_slider,
        k_slider,
        steps_slider,
        grid_slider,
        run_button,
        output,
    ])
)